# Caminho do projeto

In [2]:
import sys
from pathlib import Path

# Raiz do projeto = pasta acima de notebooks/; permite importar o pacote src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))


In [3]:
import importlib
from dotenv import load_dotenv
import src.config, src.api, src.download

# Lê o .env de novo, substituindo o valor antigo que estava na memória
load_dotenv(PROJECT_ROOT / ".env", override=True)

# Recarrega os módulos para que eles peguem a chave nova
importlib.reload(src.config)
importlib.reload(src.api)
importlib.reload(src.download)

from src.download import DownloadConfig, run_pipeline, load_daily, make_client
print("chave definida:", bool(make_client(DownloadConfig()).config.api_key))


chave definida: True


# Parâmetros e Download

In [5]:
from src.download import DownloadConfig, run_pipeline, load_daily

# Mesmos parâmetros do usgs_streamflow.ipynb
cfg = DownloadConfig(
    date_start="2010-01-01",
    date_end="2025-12-31",
    states=("California", "Minnesota", "Montana"),  # nomes por extenso, como em state_name, None = todos os estados; ou ex.: ("Ohio", "Texas")
    parameter_code="00060",        # vazão (discharge), ft3/s
    statistic_id="00003",          # média diária
    require_full_coverage=False,   # True = só séries cujo registro cobre o período inteiro
    all_columns=True,              # True = todas as colunas que a API oferece
    use_top_state=False,            # True = só o estado com mais estações
    max_sites=None,                # None = todas as estações do estado escolhido
    seed=42,                       # semente do sorteio, usada quando há max_sites
    batch_size=8,                  # séries por requisição
)

# Catálogo -> locais Stream -> seleção -> download.
# Cada série é salva em data/usgs/raw/parquets/<stem>/<time_series_id>.parquet assim que o lote termina.
# Se cair (rede, erro 429, kernel), rode esta célula de novo: o que já está em disco é pulado.
selected, summary = run_pipeline(cfg)


Catálogo: 3,718 séries; 3,698 estações
Locais do tipo Stream em 3 estado(s)
  California: 6,736 locais ST
  Minnesota: 2,602 locais ST
  Montana: 3,648 locais ST
1404 estações selecionadas em 3 estado(s)
Séries: 1404 no total, 1002 já em disco, 402 a baixar
Destino: C:\Users\aliss\OneDrive\Repositorios\usgs_streamflow\data\usgs\raw\parquets\daily_00060_00003_2010-01-01_2025-12-31
  lote 1/51: 8 séries, 38,590 linhas | 8/402 salvas | 0.1 min
  lote 2/51: 8 séries, 36,097 linhas | 16/402 salvas | 0.1 min
  lote 3/51: 8 séries, 43,950 linhas | 24/402 salvas | 0.2 min
  lote 4/51: 8 séries, 9,641 linhas | 32/402 salvas | 0.2 min
  lote 5/51: 8 séries, 46,750 linhas | 40/402 salvas | 0.2 min
  lote 6/51: 8 séries, 36,939 linhas | 48/402 salvas | 0.3 min
  lote 7/51: 8 séries, 35,764 linhas | 56/402 salvas | 0.3 min
  lote 8/51: 8 séries, 36,048 linhas | 64/402 salvas | 0.4 min
  lote 9/51: 8 séries, 39,550 linhas | 72/402 salvas | 0.4 min
  lote 10/51: 8 séries, 36,397 linhas | 80/402 salva

In [6]:
# Se houver pendentes, rode a célula 2 de novo até zerar
print(f"{len(summary['pending'])} séries pendentes | {len(summary['failures'])} lotes com falha")
for f in summary["failures"]:
    print(f"  lote {f['batch']}: {f['error']}")


0 séries pendentes | 0 lotes com falha


# Junta as séries baixadas

In [7]:
# Junta os arquivos por série que estão em disco (o mesmo `daily` do notebook anterior)
daily = load_daily(cfg, selected["time_series_id"])
print(f"{len(daily):,} observações diárias de {daily['time_series_id'].nunique()} séries")
daily.head()

6,278,372 observações diárias de 1400 séries


,time_series_id,monitoring_location_id,parameter_code,statistic_id,unit_of_measure,approval_status,qualifier,last_modified,time_api,date,value_cfs
0,164f2db2b6ea42968b33a32caaed53a4,USGS-04010500,00060,00003,ft^3/s,Approved,ESTIMATED,2025-03-11T07:28:37.876856+00:00,2013-03-02,2013-03-02,75.8
1,164f2db2b6ea42968b33a32caaed53a4,USGS-04010500,00060,00003,ft^3/s,Approved,NaN,2025-03-11T07:28:37.876856+00:00,2017-04-10,2017-04-10,1900.0
2,164f2db2b6ea42968b33a32caaed53a4,USGS-04010500,00060,00003,ft^3/s,Approved,NaN,2025-03-11T07:28:37.876856+00:00,2016-11-02,2016-11-02,188.0
3,164f2db2b6ea42968b33a32caaed53a4,USGS-04010500,00060,00003,ft^3/s,Approved,ESTIMATED,2025-03-11T07:28:37.876856+00:00,2015-12-16,2015-12-16,402.0
4,164f2db2b6ea42968b33a32caaed53a4,USGS-04010500,00060,00003,ft^3/s,Approved,ESTIMATED,2025-03-11T07:28:37.876856+00:00,2016-03-16,2016-03-16,566.0


# Monta o df_full

In [8]:
import pandas as pd

DATE_START, DATE_END = cfg.date_start, cfg.date_end
CFS_TO_M3S = 0.028316846592   # 1 ft3/s em m3/s

# IDs das séries selecionadas e calendário completo do período
series_ids = selected["time_series_id"].dropna().drop_duplicates().tolist()
calendar = pd.date_range(DATE_START, DATE_END, freq="D", name="date")

# Uma linha para cada série em cada dia, mesmo que não exista observação
grid = pd.MultiIndex.from_product(
    [series_ids, calendar],
    names=["time_series_id", "date"],
).to_frame(index=False)

# Mapeia cada série para sua estação
ts_to_site = selected.set_index("time_series_id")["monitoring_location_id"]

# Mantém as observações retornadas pela USGS
obs_cols = ["time_series_id", "date", "value_cfs", "approval_status", "qualifier"]
obs_cols = [col for col in obs_cols if col in daily.columns]
obs = daily.loc[daily["time_series_id"].isin(series_ids), obs_cols].copy()
obs["date"] = pd.to_datetime(obs["date"]).dt.normalize()

# Evita mascarar duplicatas: cada série deve ter no máximo uma linha por dia
if obs.duplicated(["time_series_id", "date"]).any():
    raise ValueError("Há observações duplicadas para uma série e uma data.")

# Marca que a linha veio da API; isso diferencia ausência de registro de valor zero
obs["record_returned"] = True

# Completa a grade: dias sem linha retornada pela API ficam com NaN
obs = grid.merge(obs, on=["time_series_id", "date"], how="left", validate="one_to_one")
obs["record_returned"] = obs["record_returned"].fillna(False).astype(bool)
obs["monitoring_location_id"] = obs["time_series_id"].map(ts_to_site)

# Converte a vazão; NaN permanece NaN e zero continua sendo uma medição válida
obs["value_m3s"] = obs["value_cfs"] * CFS_TO_M3S
obs["is_missing"] = obs["value_cfs"].isna()

# Metadados de todas as séries selecionadas, inclusive as sem observações
stations_full = selected.rename(columns={"begin": "registro_inicio", "end": "registro_fim"})
stations_full = stations_full.drop(
    columns=[
        col for col in stations_full.columns
        if col.startswith("_feature_id") or col in ("_prim", "overlaps", "full_coverage")
    ]
)

meta_cols = [
    "time_series_id", "monitoring_location_name", "state_name", "county_name",
    "lat", "lon", "drainage_area", "altitude", "site_huc", "unit_of_measure",
]
meta = stations_full[[col for col in meta_cols if col in stations_full.columns]].copy()

# Acrescenta os metadados a cada série/dia
df_full = obs.merge(meta, on="time_series_id", how="left", validate="many_to_one")

# Usa categorias para reduzir o consumo de memória
for col in (
    "time_series_id", "monitoring_location_id", "monitoring_location_name", "state_name",
    "county_name", "unit_of_measure", "approval_status", "qualifier", "site_huc",
):
    if col in df_full.columns:
        df_full[col] = df_full[col].astype("category")

# Organiza as colunas e as linhas
first_cols = [
    "date", "monitoring_location_id", "monitoring_location_name", "state_name", "county_name",
    "lat", "lon", "value_cfs", "value_m3s", "unit_of_measure", "approval_status", "qualifier",
    "record_returned", "is_missing",
]
other_cols = [col for col in df_full.columns if col not in first_cols]
df_full = df_full[[col for col in first_cols if col in df_full.columns] + other_cols]
df_full = df_full.sort_values(["monitoring_location_id", "date"]).reset_index(drop=True)

print(
    f"{len(df_full):,} linhas x {len(df_full.columns)} colunas | "
    f"{df_full['monitoring_location_id'].nunique()} estações"
)
print(f"Dias sem valor: {df_full['is_missing'].sum():,}")
print(f"Memória do df_full: {df_full.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df_full.head(10)

# Salva na mesma pasta dos downloads (cfg.out_dir = data/usgs/raw/parquets)
out = cfg.out_dir / f"df_full_{DATE_START}_{DATE_END}.parquet"
df_full.to_parquet(out)
print(f"Salvo: {out}")


8,204,976 linhas x 18 colunas | 1404 estações
Dias sem valor: 1,930,313
Memória do df_full: 0.59 GB
Salvo: C:\Users\aliss\OneDrive\Repositorios\usgs_streamflow\data\usgs\raw\parquets\df_full_2010-01-01_2025-12-31.parquet
